In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from itertools import product
import sys
import pandas as pd
from torch.utils.data import DataLoader
!rm -r /kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git clone https://github.com/KirillVishnyakov/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git -C /kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection log --oneline -1

rm: cannot remove '/kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection': No such file or directory
Cloning into 'Architectural-Biases-in-Time-Series-Anomaly-Detection'...
remote: Enumerating objects: 257, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 257 (delta 21), reused 41 (delta 13), pack-reused 207 (from 1)
Receiving objects: 100% (257/257), 2.92 MiB | 25.76 MiB/s, done.
Resolving deltas: 100% (132/132), done.
73f34b5 (HEAD -> main, origin/main, origin/HEAD) diag covarience instead since cov is inf


In [ ]:
sys.path.append("/kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection")
import dataset 
from evaluation_utils import calculate_gauss_distribution
from evaluation_utils import evaluate_lstm_scores
import lstm
if torch.cuda.is_available():
    device = torch.device("cuda")

### Set random states for reproducibility

In [4]:
seed = 432
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

### load config into environment.

In [ ]:
final_pure_forecaster_lstm = lstm.LSTMBaseline(hidden_size = 64, horizon = 4, num_layers = 1).to(device)
final_pure_forecaster_lstm.load_state_dict(torch.load("/kaggle/working/Architectural-Biases-in-Time-Series-Anomaly-Detection/saved_model_weights/best_LSTM_PURE_FORECASTER_weights.pth", weights_only=True))

<All keys matched successfully>

### Load validation, and testing datasets

In [ ]:
dummy_training_set = dataset.forecasting_Dataset(device, 100, 4, start = 0, end = 1000000, train = True)
forecasting_validation_dataset = dataset.forecasting_Dataset(device, 100, 4, dummy_training_set.scaler, start = 1000000, end = 3500000, train = False)
forecasting_testing_dataset = dataset.forecasting_Dataset(device, 100, 4, dummy_training_set.scaler, start = 3500000, end = 5000000, train = False)

In [7]:
dist = calculate_gauss_distribution(device, final_pure_forecaster_lstm, dummy_training_set)
scores, labels, categories = evaluate_lstm_scores(device, final_pure_forecaster_lstm, forecasting_validation_dataset, dist)

### Tuning the anomaly smoothing window size on separate validation data that contains anomalies

In [103]:
# Assuming 3.8% contamination rate from the dataset description
window_grid = np.linspace(1, 300, 10)
def tune_anomaly_window_size(scores, labels, window_grid, forecasting_validation_dataset):
    for window in window_grid:
        window = int(window)
        if window >= 1:
            current_score = np.convolve(scores, np.ones(window) / window, mode='same')
        # setting the anomaly to be the 100 - 3.8 th percentile, this number comes from the dataset description
        threshold = np.percentile(current_score, 100 - 3.8)
        TP = np.sum((current_score > threshold) & (labels == 1))
        FP = np.sum((current_score > threshold) & (labels == 0))
        recall = TP/forecasting_validation_dataset.total_anomalies
        precision = TP/(TP + FP)
        F1 = 2 * (precision * recall)/(precision + recall)
        print(f"| window: {window} | recall {recall:.2f}, precision {precision:.2f}, F1 {F1:.4f}")
tune_anomaly_window_size(scores, labels, window_grid, forecasting_validation_dataset)

| window: 1 | recall 0.41, precision 0.54, F1 0.4678
| window: 34 | recall 0.50, precision 0.66, F1 0.5698
| window: 67 | recall 0.53, precision 0.70, F1 0.6010
| window: 100 | recall 0.54, precision 0.71, F1 0.6110
| window: 133 | recall 0.54, precision 0.72, F1 0.6175
| window: 167 | recall 0.55, precision 0.72, F1 0.6213
| window: 200 | recall 0.55, precision 0.73, F1 0.6257
| window: 233 | recall 0.55, precision 0.73, F1 0.6257
| window: 266 | recall 0.55, precision 0.73, F1 0.6275
| window: 300 | recall 0.55, precision 0.73, F1 0.6244


### Evaluating on the test datasets

In [27]:
test_scores, test_labels, test_categories = evaluate_lstm_scores(device, final_pure_forecaster_lstm, forecasting_testing_dataset, dist)

### Evaluating F1

In [104]:
def evaluate_test_F1(scores, window, test_labels, dataset):
    current_test_score = np.convolve(scores, np.ones(window) / window, mode='same')
    threshold = np.percentile(current_test_score, 100 - 3.8)
    TP = np.sum((current_test_score > threshold) & (test_labels == 1))
    FP = np.sum((current_test_score > threshold) & (test_labels == 0))
    recall = TP/dataset.total_anomalies
    precision = TP/(TP + FP)
    F1 = 2 * (precision * recall)/(precision + recall)
    return {"evaluation_threshold": threshold.item(), "window": window, "recall" :recall.item(), "precision": precision.item(), "F1": F1.item()}
evaluate_test_F1(test_scores, 266, test_labels, forecasting_testing_dataset)
print(evaluate_test_F1(test_scores, 266, test_labels, forecasting_testing_dataset))

{'evaluation_threshold': 509.8450341580962, 'window': 266, 'recall': 0.6278660436137071, 'precision': 0.7072126603154552, 'F1': 0.6651814813898034}


### Analyzing smoothing window size contribution for recall

In [105]:
def generate_df(test_scores, test_categories, window_grid):
    series = []
    for window in window_grid:
        window = int(window)
        current_test_score = np.convolve(test_scores, np.ones(window)/window, mode='same')
        threshold = np.percentile(current_test_score, 100 - 3.8)
        cat_dict = {}
        normal_mask = test_categories == 0
        FPR = np.sum((current_test_score[normal_mask] > threshold)) / np.sum(normal_mask)
        cat_dict["FPR (0.0)"] = np.round(FPR, decimals=2).item()
    
        for cat in np.unique(test_categories):
            if cat == 0.0:
                continue
            cat_mask = test_categories == cat
            cat_TP = np.sum((current_test_score[cat_mask] > threshold))
            recall = cat_TP / np.sum(cat_mask)
            cat_dict[f"recall ({cat})"] = np.round(recall, decimals=2).item()

        cat_dict["F1"] = np.round(evaluate_test_F1(current_test_score, int(threshold), test_labels, forecasting_testing_dataset)["F1"], decimals = 3)

        series.append(pd.Series(cat_dict, name=f"t={threshold:.2f}|w={window}"))
    
    return pd.concat(series, axis=1)

generate_df(test_scores, test_categories, window_grid)

,t=307.19|w=1,t=318.10|w=34,t=352.07|w=67,t=376.12|w=100,t=400.85|w=133,t=425.92|w=167,t=452.63|w=200,t=480.78|w=233,t=509.85|w=266,t=545.39|w=300
FPR (0.0),0.020,0.010,0.010,0.010,0.010,0.010,0.010,0.010,0.010,0.010
recall (1.0),0.390,0.770,0.910,0.960,0.960,0.970,0.970,0.980,0.980,0.990
recall (2.0),0.730,0.760,0.800,0.800,0.800,0.820,0.840,0.850,0.870,0.880
recall (3.0),0.690,0.720,0.750,0.760,0.780,0.770,0.780,0.780,0.790,0.790
recall (4.0),0.410,0.410,0.410,0.410,0.410,0.420,0.420,0.430,0.440,0.450
recall (5.0),0.280,0.300,0.310,0.310,0.320,0.330,0.350,0.360,0.380,0.390
recall (6.0),0.130,0.170,0.230,0.280,0.320,0.290,0.300,0.340,0.380,0.270
recall (7.0),0.390,0.420,0.380,0.380,0.370,0.370,0.360,0.360,0.360,0.370
recall (8.0),0.640,0.680,0.710,0.730,0.760,0.780,0.770,0.790,0.770,0.780
recall (9.0),0.740,0.800,0.800,0.810,0.820,0.810,0.820,0.800,0.790,0.770


#### Model has trouble detecting anomalies: 12, 6, 13, 11, 5 (in order from hardest) the most. <br>The model is especially good at detecting anomalies: 2, 1, 3, 9. Increasing smoothing window sizes accentuates that patterns

### Check if model isnt just learning to repeat last seen window

In [83]:
import torch.nn as nn
criterion = nn.MSELoss()
def naive_baseline_loss(X, y, criterion):
    """
    Predicts y by repeating the last input timestep for all horizon steps.
    X: (batch, lookback, features)
    y: (batch, horizon, features)
    """
    last_step = X[:, -1:, :]                          
    naive_pred = last_step.expand_as(y)               
    return criterion(naive_pred, y).item()

with torch.no_grad():
    model_losses, naive_losses = [], []
    for X, y, _, _ in DataLoader(forecasting_validation_dataset, batch_size=512):
        X, y = X.to(device), y.to(device)
        pred = final_pure_forecaster_lstm(X)
        model_losses.append(criterion(pred, y).item())
        naive_losses.append(naive_baseline_loss(X, y, criterion))

print(f"Avg model loss:  {np.mean(model_losses):.4f}")
print(f"Avg naive loss:  {np.mean(naive_losses):.4f}")

Avg model loss:  0.0425
Avg naive loss:  0.0526


#### LSTM's loss is consistently lower than naive methods, meaning it is indeed learning something.

In [84]:
raw_threshold = np.percentile(test_scores, 100 - 3.8)
naive_scores = (test_scores > raw_threshold).astype(int)

from sklearn.metrics import f1_score, precision_score, recall_score
print(f"Naive F1:        {f1_score(test_labels, naive_scores):.4f}")
print(f"Naive Precision: {precision_score(test_labels, naive_scores):.4f}")
print(f"Naive Recall:    {recall_score(test_labels, naive_scores):.4f}")

Naive F1:        0.5296
Naive Precision: 0.5642
Naive Recall:    0.4991


#### Again LSTM outperforms the naive method, which means it did learn something.